# π³ SPACE · 사진에서 3D 공간으로

이 노트북이 GPU 백엔드입니다. **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하고 **런타임 → 모두 실행**하세요.

마지막 셀에서 나온 **SPACE 웹 화면 연결** 링크를 누르세요. 웹 서버가 실행 중이면 자동 연결됩니다. 또는 출력된 `https://….gradio.live` 주소를 SPACE 웹 화면의 **Colab 연결** 창에 붙여넣으세요. 그다음 웹에서 사진 2–8장을 선택하고 **3D 공간 만들기**를 누릅니다.

- 첫 실행에는 약 5.4 GB의 모델 다운로드가 필요합니다. 시간은 네트워크·GPU에 따라 달라집니다.
- 처음에는 사진 **3–4장 + 가볍게** 설정으로 시작하세요.
- Colab 탭과 런타임을 유지하세요. 런타임이 종료되면 새 주소로 다시 연결합니다.
- 사진과 결과는 이 Colab에 저장됩니다. 작업 후 **연결 해제 및 런타임 삭제**로 정리할 수 있습니다.
- 공개 주소를 아는 사람은 API를 사용할 수 있습니다.
- 결과는 색이 있는 점 구름입니다. **Pi3X 가중치는 비상업 연구·교육용**입니다.

[공식 Pi3X 코드](https://github.com/yyfz/Pi3) · [모델](https://huggingface.co/yyfz233/Pi3X)


## 1. GPU 확인과 준비
필요한 패키지만 Colab에 설치합니다. 기존 PyTorch와 CUDA를 유지합니다.


In [ ]:
# 이 셀은 Colab에서만 실행하세요. 로컬에는 설치하지 않습니다.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if not Path("/content").exists():
    raise RuntimeError("Google Colab에서 실행해주세요.")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("런타임 → 런타임 유형 변경 → T4 GPU를 선택한 뒤 다시 실행하세요.")
print("GPU:", torch.cuda.get_device_name(0))
print("기존 PyTorch 사용:", torch.__version__)

# torch / torchvision / CUDA / numpy를 재설치하거나 다운그레이드하지 않습니다.
# Pi3X의 image-only 추론에는 아래 패키지만 추가로 필요합니다.
packages = ["gradio==5.49.1", "huggingface_hub==0.35.3", "safetensors==0.6.2"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
for module in ("numpy", "PIL"):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f"Colab 기본 패키지 {module}가 없습니다. 새 GPU 런타임에서 다시 실행하세요.")

# 공식 소스 버전을 고정해 이후 upstream 변경의 영향을 줄입니다.
SOURCE_REVISION = "9fa3ddb3f8d53041f8b2738df404f62223bbaa7b"
repository = Path("/content/Pi3_SPACE_source")
if not (repository / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/yyfz/Pi3.git", str(repository)], check=True)
subprocess.run(["git", "-C", str(repository), "checkout", "--detach", SOURCE_REVISION], check=True)
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
os.environ["XFORMERS_DISABLED"] = "1"
print("준비 완료! GPU 서버를 시작합니다. 셀이 나뉘어 있으면 다음 셀을 실행하세요.")


## 2. GPU 서버 시작
GitHub Pages에서는 배포된 웹 주소를 사용하세요. 로컬 미리보기에서는 VS Code에서 먼저 `python3 start.py`를 실행하세요. 셀의 `VIEWER_URL` 값을 실제 브라우저에서 사용하는 주소로 확인해주세요. 이 셀을 실행한 뒤 **SPACE 웹 화면 연결** 링크를 누르면 자동으로 연결합니다. 웹 연결이 어려우면 이 셀의 Gradio 화면에서도 사진을 올려 PLY를 만들 수 있습니다. 받은 PLY를 SPACE의 폴더 버튼으로 불러오세요.


In [ ]:
"""Pi3X SPACE GPU backend. Run in Google Colab, never on the local viewer server.

Model source: https://github.com/yyfz/Pi3
Weights: https://huggingface.co/yyfz233/Pi3X (CC BY-NC 4.0).
"""
import gc
import math
import os
import shutil
import tempfile
import time
from html import escape
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

# Force the repository's PyTorch attention fallback; no compiled extension installs.
os.environ["XFORMERS_DISABLED"] = "1"

import gradio as gr
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from PIL import Image, ImageOps, UnidentifiedImageError
from safetensors import safe_open

from pi3.models.pi3x import Pi3X
from pi3.utils.geometry import depth_normal_edge

WEIGHTS_REVISION = "bb1deea4d7423de5b30691739cb451a3f57dc1d5"
PIXEL_LIMITS = {"fast": 90000, "detail": 230000}
OUTPUT_ROOT = Path("/content/SPACE_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL = None


def load_model(progress):
    global MODEL
    if MODEL is not None:
        return MODEL
    progress(.10, desc="Pi3X 모델을 내려받고 있어요. 첫 실행은 몇 분 걸릴 수 있어요.")
    checkpoint = hf_hub_download(
        repo_id="yyfz233/Pi3X", filename="model.safetensors", revision=WEIGHTS_REVISION
    )
    # Instantiate on meta to avoid two copies of the model in Colab's host RAM.
    # Only image-only branches are needed for this application.
    with torch.device("meta"):
        candidate = Pi3X(use_multimodal=False).eval()
    expected = set(candidate.state_dict())
    with safe_open(checkpoint, framework="pt", device="cpu") as handle:
        available = set(handle.keys())
        missing = expected - available
        if missing:
            raise RuntimeError("Pi3X checkpoint is missing model tensors: " + str(sorted(missing)[:5]))
        tensors = {key: handle.get_tensor(key) for key in expected}
    candidate.load_state_dict(tensors, strict=True, assign=True)
    del tensors
    progress(.22, desc="모델을 GPU에 준비하고 있어요.")
    candidate = candidate.to("cuda").eval()
    MODEL = candidate
    gc.collect()
    return MODEL


def load_photos(paths, pixel_limit):
    photos = []
    for path in paths:
        path = Path(path)
        if path.stat().st_size > 15 * 1024 * 1024:
            raise gr.Error("사진은 장당 15 MB 이하로 선택해주세요.")
        try:
            with Image.open(path) as source:
                if source.format not in {"JPEG", "PNG", "WEBP"}:
                    raise gr.Error("JPG, PNG, WEBP 사진만 지원합니다.")
                if min(source.size) < 28 or max(source.size) / min(source.size) > 6:
                    raise gr.Error("너무 작거나 긴 사진입니다. 일반적인 가로·세로 사진을 사용해주세요.")
                photos.append(ImageOps.exif_transpose(source).convert("RGB"))
        except (UnidentifiedImageError, OSError, Image.DecompressionBombError):
            raise gr.Error("읽을 수 없는 사진입니다. 파일을 확인해주세요.") from None
    width, height = photos[0].size
    scale = math.sqrt(pixel_limit / (width * height))
    cols, rows = max(1, round(width * scale / 14)), max(1, round(height * scale / 14))
    while cols * rows * 196 > pixel_limit:
        if cols / rows > width / height and cols > 1:
            cols -= 1
        elif rows > 1:
            rows -= 1
        else:
            cols -= 1
    target = (cols * 14, rows * 14)
    rgb = np.stack([np.asarray(photo.resize(target, Image.Resampling.LANCZOS), dtype=np.uint8) for photo in photos])
    tensor = torch.from_numpy(rgb.copy()).permute(0, 3, 1, 2).float().div_(255).unsqueeze(0).to("cuda")
    return tensor, rgb, target


def save_cloud(path, points, colors, confidence):
    """Packed binary PLY with an extra confidence property for browser filtering."""
    count = len(points)
    records = np.empty(count, dtype=[("x", "<f4"), ("y", "<f4"), ("z", "<f4"),
                                  ("red", "u1"), ("green", "u1"), ("blue", "u1"),
                                  ("confidence", "<f4")])
    for i, key in enumerate(("x", "y", "z")):
        records[key] = points[:, i]
    for i, key in enumerate(("red", "green", "blue")):
        records[key] = colors[:, i]
    records["confidence"] = confidence
    header = ("ply\nformat binary_little_endian 1.0\n"
              "comment Pi3X SPACE; OpenCV camera-to-world coordinates\n"
              f"element vertex {count}\nproperty float x\nproperty float y\nproperty float z\n"
              "property uchar red\nproperty uchar green\nproperty uchar blue\n"
              "property float confidence\nend_header\n")
    with open(path, "wb") as stream:
        stream.write(header.encode("ascii"))
        records.tofile(stream)


def clean_old_results():
    # Keep results long enough for viewing/downloading, without unbounded disk growth.
    for directory in OUTPUT_ROOT.glob("scene_*"):
        if directory.is_dir() and directory.stat().st_mtime < time.time() - 6 * 3600:
            shutil.rmtree(directory, ignore_errors=True)


def reconstruct(images, quality, progress=gr.Progress()):
    if not torch.cuda.is_available():
        raise gr.Error("Colab에서 런타임 → 런타임 유형 변경 → T4 GPU를 선택해주세요.")
    if not images or not 2 <= len(images) <= 8:
        raise gr.Error("같은 공간의 사진 2–8장을 선택해주세요.")
    if quality not in PIXEL_LIMITS:
        raise gr.Error("지원하지 않는 품질 설정입니다.")
    started = time.monotonic()
    tensor = result = masks = model = None
    try:
        clean_old_results()
        model = load_model(progress)
        progress(.30, desc="사진의 크기와 방향을 맞추고 있어요.")
        tensor, rgb, target = load_photos(images, PIXEL_LIMITS[quality])
        progress(.42, desc="Pi3X가 사진을 연결해 3D 공간을 재구성하고 있어요.")
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        with torch.inference_mode(), torch.amp.autocast("cuda", dtype=dtype):
            result = model(imgs=tensor)
        progress(.82, desc="경계의 노이즈를 정리하고 점 구름을 만들고 있어요.")
        scores = torch.sigmoid(result["conf"][..., 0]).float()
        # Preserve low confidence values so the user can adjust filtering live.
        masks = (scores >= .01) & torch.isfinite(result["points"]).all(dim=-1)
        masks &= ~depth_normal_edge(result["local_points"], rtol=.03, mask=masks)
        selection = masks[0].cpu().numpy()
        points = result["points"][0].float().cpu().numpy()[selection]
        confidence = scores[0].cpu().numpy()[selection]
        colors = rgb[selection]
        if not len(points):
            raise gr.Error("유효한 점이 없어요. 겹치는 영역이 더 많은 선명한 사진으로 시도해주세요.")
        if len(points) > 600000:
            indices = np.random.default_rng(0).choice(len(points), 600000, replace=False)
            points, colors, confidence = points[indices], colors[indices], confidence[indices]
        directory = Path(tempfile.mkdtemp(prefix="scene_", dir=OUTPUT_ROOT))
        output = directory / "SPACE_reconstruction.ply"
        save_cloud(output, points, colors, confidence)
        metadata = {"engine": "Pi3X", "image_count": len(images), "point_count": len(points),
                    "resolution": list(target), "quality": quality,
                    "elapsed_seconds": round(time.monotonic() - started, 1),
                    "coordinate_system": "OpenCV", "weights_revision": WEIGHTS_REVISION}
        progress(1, desc="완성! 브라우저로 공간을 보내고 있어요.")
        return str(output), metadata
    except torch.cuda.OutOfMemoryError:
        raise gr.Error("GPU 메모리가 부족해요. ‘가볍게’로 바꾸거나 사진을 2–4장으로 줄여주세요.") from None
    finally:
        tensor = result = masks = None
        gc.collect()
        torch.cuda.empty_cache()


def viewer_connection_url(viewer_url, public_url):
    """Return a click-to-connect browser URL without persisting any credentials."""
    parts = urlsplit(viewer_url)
    if parts.scheme not in {"http", "https"} or not parts.netloc or parts.username or parts.password:
        raise ValueError("웹 화면 주소에는 http:// 또는 https:// 주소를 입력해주세요.")
    query = [(key, value) for key, value in parse_qsl(parts.query) if key != "colab"]
    query.append(("colab", public_url))
    return urlunsplit(parts._replace(query=urlencode(query)))


def launch(viewer_url="http://localhost:8000/"):
    if not torch.cuda.is_available():
        raise RuntimeError("GPU가 연결되지 않았습니다. 런타임 유형을 T4 GPU로 변경하고 다시 실행하세요.")
    with gr.Blocks(title="Pi3X SPACE · Colab GPU") as demo:
        gr.Markdown("# π³ SPACE · Colab GPU\n이 탭을 열어두고 아래 공개 주소를 SPACE 웹 화면에 붙여넣으세요.\n"
                    "사진은 이 Colab 런타임에서 처리됩니다. 모델 가중치는 비상업 연구·교육용입니다.")
        images = gr.File(label="같은 공간의 사진 2–8장", file_count="multiple", type="filepath", file_types=["image"])
        quality = gr.Radio(choices=[("가볍게", "fast"), ("섬세하게", "detail")], value="fast", label="품질")
        run = gr.Button("3D 공간 만들기")
        output = gr.File(label="PLY 결과")
        metadata = gr.JSON(label="재구성 정보")
        run.click(reconstruct, inputs=[images, quality], outputs=[output, metadata],
                  api_name="reconstruct", concurrency_limit=1)
    demo.queue(max_size=4, default_concurrency_limit=1)
    _, _, public_url = demo.launch(share=True, debug=False, show_error=True, max_file_size="15mb")
    print("\n" + "=" * 50)
    print("SPACE 웹 화면에 붙여넣을 주소:", public_url)
    print("이 셀과 Colab 런타임을 실행 상태로 유지하세요.")
    print("=" * 50)
    if public_url:
        try:
            connection_url = viewer_connection_url(viewer_url, public_url)
            from IPython.display import HTML, display
            display(HTML(
                '<a target="_blank" rel="noopener" href="' + escape(connection_url, quote=True)
                + '" style="display:inline-block;padding:14px 22px;border-radius:8px;'
                + 'background:#baff6c;color:#1d3412;text-decoration:none;font-weight:700">'
                + 'SPACE 웹 화면 연결</a>'
                + '<p>이 링크를 누르면 SPACE 웹 화면이 열리고 Colab GPU에 연결됩니다.</p>'
            ))
        except (ValueError, ImportError) as error:
            print("웹 자동 연결 링크를 만들지 못했습니다:", error)
            print("위 gradio.live 주소를 웹 화면의 Colab 연결 창에 붙여넣으세요.")
    return demo



# 실행 후 돌아갈 SPACE 주소. GitHub Pages를 쓰면 배포한 웹 주소를 넣으세요.
VIEWER_URL = "http://localhost:8000/" # @param {type:"string"}

# 셀을 다시 실행하면 이전 Gradio 서버를 닫습니다.
gr.close_all()
space_demo = launch(VIEWER_URL)
